# 39 · HuggingFace 生态 + chat template + 推理基础

> **学习目标**：理解 transformers 三件套 / chat template / generate 参数；手撸 apply_chat_template → generate 全流程；对比 fp16/bf16/int4/ GGUF。

> **预备**：00-基础 Stage 1-2 走完（了解 numpy / hash / markdown）。

> **为什么重要**：所有微调工作都基于 HF 生态。**不懂 chat template 就别开始 SFT** —— 模型不会知道你在跟它对话，loss 看起来降但什么都没学到。

**注意**：本 notebook 所有 LLM 调用都用 OFFLINE stub，训练部分需要切换到 `ft` env。

In [ ]:
MODE = 'OFFLINE'  # 'OFFLINE'（离线）或 'ONLINE'（在线，需要 ft env）

import json, hashlib, time
print(f'MODE = {MODE}')

## 1. transformers 三件套 —— AutoModel / AutoTokenizer / pipeline

三件套统一入口：`AutoModel` / `AutoTokenizer` / `pipeline`。

**关键差异**：
- `AutoModel` / `AutoTokenizer` → **加载时全部在内存**，适合微调
- `pipeline` → **懒加载**，用的时候才下载模型，适合推理

**本节 OFFLINE 模式**：用 dict 模拟 model output，避免真实加载。

In [ ]:
# OFFLINE 模式：fake model / tokenizer

class FakeTokenizer:
    def __init__(self, vocab_size=10000):
        self.vocab_size = vocab_size
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=True, **kwargs):
        """
        模拟 apply_chat_template：把对话列表拼接成单个 prompt。
        实际调用：tokenizer.apply_chat_template(messages, tokenize=False, ...)  # 返回 str
        """
        # 实际 tokenizer 返回 list[int] 如果 tokenize=True
        if tokenize:
            return [self.vocab_size + i for i, m in enumerate(messages)]
        prompt = ''
        for i, m in enumerate(messages):
            role = m.get('role', 'user')
            content = m.get('content', '')
            # 模拟 OpenAI 格式
            prompt += f'{role}: {content}\n'
        if add_generation_prompt:
            prompt += 'assistant: '
        return prompt

class FakeModel:
    def __init__(self, dim=4096):
        self.dim = dim
    def generate(self, input_ids, max_new_tokens=50, temperature=0.7, do_sample=True, **kwargs):
        """
        模拟 model.generate：只返回固定的假回答。
        实际调用：model.generate(input_ids, max_new_tokens=50, temperature=0.7, do_sample=True)
        """
        fake_response = '[FAKE-MODEL] 基于上下文我能告诉你: 这是模型生成的回答。'
        return fake_response

# 模拟 HF 统一接口
def AutoModel.from_pretrained(model_name_or_path, **kwargs):
    return FakeModel(**kwargs)

def AutoTokenizer.from_pretrained(model_name_or_path, **kwargs):
    return FakeTokenizer()

# 模拟 pipeline（懒加载）
def pipeline(task, model, tokenizer, **kwargs):
    return lambda **kw: model.generate(**kw)

## 2. chat template 实战 —— apply_chat_template 的作用

**chat template** 把「对话列表」转成「单次推理的 prompt」。

**作用**：
1. 告诉模型「上下文」和「当前轮次」
2. 强制统一的对话格式（OpenAI / Qwen / Llama 格式不同）
3. 避免「模型自己发明格式」导致性能腰斩

**关键**：SFT 必须用 `apply_chat_template`，否则模型不知道你在跟它对话 —— loss 降但什么都没学到。

In [ ]:
# 构建 2 轮对话
messages = [
    {'role': 'user', 'content': '你好，请介绍一下自己。'},
    {'role': 'assistant', 'content': '你好！我是一个 AI 助手，可以帮助你回答问题。'},
    {'role': 'user', 'content': '那你能做什么？'}
]

# OFFLINE 模式：用 FakeTokenizer 模拟
tokenizer = FakeTokenizer()

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print('=== prompt（单次推理）===')
print(prompt)
print(f'\n长度: {len(prompt)} 字符')

In [ ]:
# 对比：如果不用 chat template，直接把所有 messages 拼上去，会怎样？
naive_prompt = '\n'.join([f'{m["role"]}: {m["content"]}' for m in messages]) + '\nassistant: '

print('=== naive prompt（没用 template）===')
print(naive_prompt)
print(f'\n长度: {len(naive_prompt)} 字符')

print('\n**关键差异**：')
print('  - chat template 会添加 `<|im_start|>assistant<|im_end|>` 等特殊 token（取决于模型家族）')
print('  - chat template 保证模型知道「当前轮次」是 assistant，而不是 user')
print('  - 不同模型（Qwen / Llama / Mistral）的 template 完全不同，必须用各自的）')

## 3. generate 参数解密 —— max_new_tokens / temperature / do_sample

三件套是「骨架」，`generate` 参数决定「生成风格」。

**关键参数**：
- `max_new_tokens`：限制输出长度（防止无限生成）
- `temperature`：控制随机性（0 = 确定生成，大 = 更多样）
- `do_sample`：是否采样（True = 从概率分布采样，False = 取 argmax）

In [ ]:
# 用 OFFLINE 模型测试不同参数组合
model = FakeModel()

print('=== do_sample=False (取 argmax) ===')
for max_new in [20, 50, 100]:
    out = model.generate(
        input_ids=[1, 2, 3],
        max_new_tokens=max_new,
        do_sample=False,
        temperature=1.0
    )
    print(f'  max_new_tokens={max_new:3d} -> 长度: {len(out)} 字符')

print('\n=== do_sample=True (采样) ===')
for temp in [0.1, 0.5, 1.0, 1.5]:
    out = model.generate(
        input_ids=[1, 2, 3],
        max_new_tokens=30,
        do_sample=True,
        temperature=temp
    )
    print(f'  temperature={temp:.1f} -> 输出: {out[:50]}...')

## 4. 量化认知 —— fp16 / bf16 / int8 / int4 / GGUF

**精度 vs 显存权衡**：

| 量化级别 | 精度 | 显存收益 | 典型用途 |
|---------|------|---------|---------|
| fp16    | 半精度 float16 | 基准 | 大多数推理 |
| bf16    | bfloat16（8bit 指数，不损失动态范围） | 与 fp16 相当 | 训练友好，大模型首选 |
| int8    | 8bit 整数 | 2x 收益 | 中等显存需求 |
| int4    | 4bit 整数 | 4x 收益 | 7B+ 模型微调 / 推理 |
| GGUF    | 4-8bit int（专为 llama.cpp/Ollama 设计） | 4x 收益 | CPU / 本地部署 |

**为什么 GGUF 跑得比 safetensors 快**？
- GGUF 是专为量化 + CPU 推理优化的格式
- safetensors 是用于训练和 fp16/bf16 推理的原始格式
- GGUF 内部做了更多 layout 优化（冻结层、分组量化）

In [ ]:
# OFFLINE 模式：模拟不同精度的显存占用

def simulate_memory(model_size_gb, quant_bits):
    """模拟不同量化级别的显存占用（仅供参考）"""
    # 基准：fp16 (2 bytes per parameter)
    fp16_size_gb = model_size_gb  # 假设输入已经是 fp16
    
    if quant_bits == 16:
        return fp16_size_gb
    elif quant_bits == 8:
        return fp16_size_gb / 2
    elif quant_bits == 4:
        return fp16_size_gb / 4
    else:
        return fp16_size_gb

# 对比 7B 模型的不同精度
model_sizes = {
    '7B (fp16)': 7.0,
    '7B (bf16)': 7.0,
    '7B (int8)': 7.0 / 2,
    '7B (int4)': 7.0 / 4,
    '7B (GGUF)': 7.0 / 4,
}

print('=== 7B 模型不同精度显存占用 ===')
for name, size_gb in model_sizes.items():
    print(f'  {name:15s} -> {size_gb:.2f} GB')

# 对比推理速度（纯文本，非真实 benchmark）
print('\n=== 推理速度对比（7B）===')
print('  fp16:    1x (基准)')
print('  bf16:    ~1x')
print('  int8:    ~0.9x')
print('  int4:    ~0.6-0.7x')
print('  GGUF:    ~0.4-0.5x (CPU) / ~0.8x (GPU)')

## 深入思考

1. **为什么 SFT 一定要用 chat template？**
   - 没有 template，模型不知道「上下文」和「当前轮次」，会自己发明格式 → 性能腰斩。
2. **temperature=0 生成会不会太机械？**
   - 生成任务常用 0（避免多样性）。问答任务可以用 0.7-0.9。调优很重要。
3. **bf16 为什么更适合训练？**
   - bf16 的指数是 8bit，精度 7bit —— 比 fp16 的 10bit 精度更稳，大模型训练不爆炸。
4. **int4 训练会怎样？**
   - int4 **不直接**用于训练（训练需要 fp16/bf16）。int4 只能用于推理或 **PEFT**（LoRA/QLoRA）。
5. **GGUF 能用于微调吗？**
   - **不能**。GGUF 是专为推理设计的格式，训练要用 safetensors / pytorch bin。

**改一改**：
 - 在 REAL env 里，`tokenizer.apply_chat_template` 会返回 `list[int]`（token ids）而不是 str
 - `model.generate` 返回 `torch.Tensor`，需要 `.decode()` 解成文本
 - 加一个真实 chat template 示例（如 Qwen 的 `<|im_start|>` 格式）

## 自检 ✅

- [ ] 默写 transformers 三件套的统一入口函数名
- [ ] 解释 chat template 的 3 个作用
- [ ] 画出 generate 参数（max_new_tokens / temperature / do_sample）如何影响输出
- [ ] 解释 fp16 / bf16 / int8 / int4 各自精度损失与显存收益
- [ ] 解释「为什么 GGUF 跑得比 safetensors 快」

## 下一步

→ [`40_lora_qlora_sft.ipynb`](40_lora_qlora_sft.ipynb)